# `matrix_function_backward_1` — Interactive Visualization

**Forward pass:** `N = X @ W`, then `S = σ(N)`

**Backward pass (gradient w.r.t. X):**
$$\frac{dS}{dX} = \frac{dS}{dN} \cdot \frac{dN}{dX} = \sigma'(N) \cdot W^T$$

Use the controls below to change matrix shapes, the activation function, and the random seed.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
import ipywidgets as widgets
from IPython.display import display
import sys, os

sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

# ── Activation functions ────────────────────────────────────────────────────
def sigmoid(x):   return 1 / (1 + np.exp(-x))
def relu(x):      return np.maximum(0, x)
def tanh(x):      return np.tanh(x)
def leaky_relu(x): return np.where(x > 0, x, 0.01 * x)
def linear(x):    return x

SIGMAS = {"sigmoid": sigmoid, "tanh": tanh, "relu": relu,
          "leaky_relu": leaky_relu, "linear": linear}

# ── Numerical derivative (same as derivative.py) ────────────────────────────
def deriv(func, input_, delta=0.001):
    return (func(input_ + delta) - func(input_ - delta)) / (2 * delta)

# ── Core function (mirrors matrix_function_backward_1.py) ───────────────────
def matrix_function_backward_1(X, W, sigma):
    assert X.shape[1] == W.shape[0]
    N    = X @ W
    S    = sigma(N)
    dSdN = deriv(sigma, N)
    dNdX = W.T
    grad = dSdN @ dNdX
    return N, S, dSdN, dNdX, grad

print("Setup complete.")

Setup complete.


In [2]:
def heatmap(ax, data, title, fmt=".2f", cmap="RdBu_r", center=None):
    vmax = np.abs(data).max() or 1.0
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax) if center == 0 else None
    im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto", interpolation="nearest")
    plt.colorbar(im, ax=ax, shrink=0.8)
    rows, cols = data.shape
    for r in range(rows):
        for c in range(cols):
            val = data[r, c]
            color = "white" if abs(val) > 0.6 * vmax else "black"
            ax.text(c, r, f"{val:{fmt}}", ha="center", va="center",
                    fontsize=max(5, 9 - max(rows, cols)), color=color)
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xticks(range(cols)); ax.set_yticks(range(rows))
    ax.set_xticklabels([f"c{i}" for i in range(cols)], fontsize=8)
    ax.set_yticklabels([f"r{i}" for i in range(rows)], fontsize=8)

def draw(batch, in_feat, out_feat, sigma_name, seed):
    sigma = SIGMAS[sigma_name]
    rng   = np.random.default_rng(seed)
    X     = rng.standard_normal((batch, in_feat))
    W     = rng.standard_normal((in_feat, out_feat))

    N, S, dSdN, dNdX, grad = matrix_function_backward_1(X, W, sigma)

    # curve domain: centre on actual N values with some padding
    N_flat  = N.flatten()
    pad     = max(2.5, np.abs(N_flat).max() * 0.4)
    xs      = np.linspace(N_flat.min() - pad, N_flat.max() + pad, 500)
    ys      = sigma(xs)
    dys     = deriv(sigma, xs)

    fig = plt.figure(figsize=(16, 16))
    fig.suptitle(
        f"matrix_function_backward_1  |  σ = {sigma_name}  |  "
        f"X:{X.shape}  W:{W.shape}  → grad:{grad.shape}",
        fontsize=13, fontweight="bold", y=0.995
    )
    gs = gridspec.GridSpec(4, 4, figure=fig, hspace=0.65, wspace=0.45)

    # ── Row 0: inputs ─────────────────────────────────────────────────────
    heatmap(fig.add_subplot(gs[0, 0:2]), X,    f"X  {X.shape}\n(input)",   center=0)
    heatmap(fig.add_subplot(gs[0, 2:4]), W,    f"W  {W.shape}\n(weights)", center=0)

    # ── Row 1: forward pass ───────────────────────────────────────────────
    heatmap(fig.add_subplot(gs[1, 0:2]), N,    f"N = X @ W  {N.shape}\n(pre-activation)", center=0)
    heatmap(fig.add_subplot(gs[1, 2:4]), S,    f"S = σ(N)  {S.shape}\n(post-activation)")

    # ── Row 2: backward pass ──────────────────────────────────────────────
    heatmap(fig.add_subplot(gs[2, 0]),   dSdN, f"dS/dN = σ'(N)  {dSdN.shape}")
    heatmap(fig.add_subplot(gs[2, 1]),   dNdX, f"dN/dX = Wᵀ  {dNdX.shape}", center=0)

    ax_eq = fig.add_subplot(gs[2, 2])
    ax_eq.axis("off")
    ax_eq.text(0.5, 0.5, "dS/dX =\ndS/dN @ dN/dX\n= σ'(N) @ Wᵀ",
               ha="center", va="center", fontsize=12,
               bbox=dict(boxstyle="round,pad=0.5", fc="#f0f4ff", ec="#99aacc"))

    heatmap(fig.add_subplot(gs[2, 3]),   grad, f"GRADIENT  dS/dX  {grad.shape}", center=0)

    # ── Row 3: σ and σ' curves with actual N values overlaid ─────────────
    ax_s  = fig.add_subplot(gs[3, 0:2])
    ax_ds = fig.add_subplot(gs[3, 2:4])

    # σ(x) curve
    ax_s.plot(xs, ys, color="#2563eb", lw=2.5, label=f"σ(x)")
    # scatter actual N values on the curve
    y_pts_s = sigma(N_flat)
    ax_s.scatter(N_flat, y_pts_s, color="#dc2626", s=55, zorder=5,
                 label=f"N values  (n={len(N_flat)})")
    for xv, yv in zip(N_flat, y_pts_s):
        ax_s.vlines(xv, 0, yv, color="#dc2626", lw=0.7, alpha=0.35, linestyle="--")
    ax_s.set_title(f"σ(x) — activation  [{sigma_name}]", fontsize=11, fontweight="bold")
    ax_s.set_xlabel("x  (pre-activation N values)", fontsize=9)
    ax_s.set_ylabel("σ(x)", fontsize=9)
    ax_s.legend(fontsize=9); ax_s.grid(True, alpha=0.3)

    # σ'(x) curve
    ax_ds.plot(xs, dys, color="#16a34a", lw=2.5, label="σ'(x)  (gradient)")
    ax_ds.axhline(0, color="#9ca3af", lw=0.8, ls="--")
    y_pts_d = deriv(sigma, N_flat)
    ax_ds.scatter(N_flat, y_pts_d, color="#dc2626", s=55, zorder=5,
                  label=f"σ'(N)  → used as dS/dN")
    for xv, yv in zip(N_flat, y_pts_d):
        ax_ds.vlines(xv, 0, yv, color="#dc2626", lw=0.7, alpha=0.35, linestyle="--")
    ax_ds.set_title("σ'(x) — derivative / gradient signal", fontsize=11, fontweight="bold")
    ax_ds.set_xlabel("x  (pre-activation N values)", fontsize=9)
    ax_ds.set_ylabel("σ'(x)", fontsize=9)
    ax_ds.legend(fontsize=9); ax_ds.grid(True, alpha=0.3)

    plt.show()
    print(f"Shapes — X:{X.shape}  W:{W.shape}  N:{N.shape}  "
          f"dSdN:{dSdN.shape}  dNdX:{dNdX.shape}  grad:{grad.shape}")

print("Plotting helpers ready.")

Plotting helpers ready.


In [3]:
w_batch    = widgets.IntSlider(value=3, min=1, max=8,  description="batch (rows X)",   style={"description_width":"140px"}, layout=widgets.Layout(width="420px"))
w_in_feat  = widgets.IntSlider(value=4, min=1, max=8,  description="in_feat (cols X = rows W)", style={"description_width":"140px"}, layout=widgets.Layout(width="420px"))
w_out_feat = widgets.IntSlider(value=2, min=1, max=8,  description="out_feat (cols W)", style={"description_width":"140px"}, layout=widgets.Layout(width="420px"))
w_sigma    = widgets.Dropdown(options=list(SIGMAS.keys()), value="sigmoid",
                              description="σ (activation)", style={"description_width":"140px"})
w_seed     = widgets.IntSlider(value=42, min=0, max=999, description="random seed",
                               style={"description_width":"140px"}, layout=widgets.Layout(width="420px"))

ui = widgets.VBox([
    widgets.HTML("<b>Matrix dimensions</b>"),
    w_batch, w_in_feat, w_out_feat,
    widgets.HTML("<b>Activation &amp; randomness</b>"),
    w_sigma, w_seed,
])

out = widgets.interactive_output(
    draw,
    {"batch": w_batch, "in_feat": w_in_feat, "out_feat": w_out_feat,
     "sigma_name": w_sigma, "seed": w_seed}
)

display(ui, out)

Output()